# The token tax of handoffs

**Scenario:** a supervisor service watches climate ensemble runs. A node log arrives, and six steps
turn it into an operator handover. Six small questions, six small answers, one alarming invoice.

The steps did not get bigger. The thing being carried between them did.

Think of it as a relay where every runner carries the whole track. The baton is tiny. Nobody is
carrying the baton.

## Mechanics

A chat call has no memory. Each step sends the whole message list again, and every token in it is
charged again at the prompt rate.

| Piece of the list | Sent on | Charged |
|---|---|---|
| System message | Every step | Prompt rate, every time |
| The run log | Every step, once it is in the list | Prompt rate, every time |
| Each earlier question | Every step after it | Prompt rate, every time |
| Each earlier answer | Every step after it | Prompt rate, every time |
| The new question | Its own step | Prompt rate, once |

So the prefix, meaning the unchanging start of every request, is not paid for once. It is paid for
as many times as you make a call.

## The picture

![Each step resends the log, until a digest replaces it](images/handoff-tax.svg)

Only the first step needs the raw log. Every later step is paying for it out of habit.

## The cost

```
prompt_bill = sum over steps of every token in the list at that step
paid_again  = (steps - 1) * tokens in step one's list
```

The second line is exact, not an estimate. Step one's list is still there at step six, so every
token of it was charged once per later step.

## The failure

The log is 24 ensemble members, one line each, and six questions an operator would ask about it.

In [1]:
BRIEF = ("Ensemble CMIP-X run 2041-B, 24 members, 0.25 degree grid.\nNode log:\n" + "\n".join(
    f"node-{i:02d} member={i} status={'ok' if i % 7 else 'stalled'}"
    f" wall={3000 + i * 37}s ckpt=t+{i * 4}h drift={0.01 * i:.2f}K" for i in range(24)))

STEPS = ["Summarise the run state in two lines.",
         "Which member ids need re-running?",
         "What is the wallclock risk to the deadline? Two lines.",
         "Which variables are most affected? Two lines.",
         "Draft one line for the operator handover note.",
         "Recommend one action. One sentence."]

SYSTEM = "You supervise climate ensemble runs."

The chain is written the ordinary way. Append the question, call, append the answer, move on. That
growing list is the only state the service keeps.

In [2]:
from vault import Usage, get_client, load_env, model_for, summarise

load_env()
MODEL = model_for("default")
client = get_client("03-token-economics/02-the-token-tax-of-handoffs")


def ask(messages, question):
    """One step. The whole list goes over the wire, every time."""
    messages.append({"role": "user", "content": question})
    reply = client.chat.completions.create(model=MODEL, max_tokens=200, messages=messages)
    messages.append({"role": "assistant", "content": reply.choices[0].message.content})
    return Usage.from_response(reply)

Run all six and watch the prompt count per step. The questions get shorter as the chain goes on.

In [3]:
history = [{"role": "system", "content": SYSTEM},
           {"role": "user", "content": BRIEF}]
naive = [ask(history, question) for question in STEPS]

for i, usage in enumerate(naive, start=1):
    print(f"  step {i}: prompt {usage.prompt_tokens:5}   completion {usage.completion_tokens:4}")

print(f"\ntotal prompt tokens: {summarise(naive)['prompt_tokens']}")

  step 1: prompt   855   completion  102
  step 2: prompt   965   completion   58
  step 3: prompt  1036   completion   49
  step 4: prompt  1094   completion   51
  step 5: prompt  1154   completion   48
  step 6: prompt  1209   completion   39

total prompt tokens: 6313


Nothing crashed, so this looks fine on a dashboard. The last step asks a shorter question than the
first, so it should not be the dearest call.

In [4]:
first, last = naive[0].prompt_tokens, naive[-1].prompt_tokens
paid = summarise(naive)["prompt_tokens"]
paid_again = (len(naive) - 1) * first

print(f"step 1 prompt tokens : {first}")
print(f"step 6 prompt tokens : {last}")
print(f"resent from step 1   : {paid_again} of {paid}, {paid_again / paid:.0%} of the prompt bill")

assert last <= first, "the last step asks a shorter question and costs more than the first"

step 1 prompt tokens : 855
step 6 prompt tokens : 1209
resent from step 1   : 4275 of 6313, 68% of the prompt bill


AssertionError: the last step asks a shorter question and costs more than the first

## The diagnosis

The assertion fires, and the third line says why.

**The list is the request.** There is no session on the far side. Every step rebuilds the same
prefix and pays the prompt rate on all of it again.

**The log is carried by every step and read by one.** Only the first question needs 24 node lines.
The other five are charged for them anyway.

**Growth compounds with chain length.** Each answer joins the list, so step six carries five answers
it will never quote.

This is not a leak. It is the ordinary shape of a chain, and it grows with every step you add.

## The fix

Pay for the log once. Turn it into the small thing the later steps actually need, then hand that
along instead.

In [5]:
CARRY_RULE = ("Compress the log into at most 60 words, for a colleague who will not see it. "
              "Keep every stalled member id and the worst drift value.")


def make_carry(brief):
    """One call that reads the log, so no later step has to."""
    reply = client.chat.completions.create(
        model=MODEL, max_tokens=200,
        messages=[{"role": "system", "content": SYSTEM},
                  {"role": "user", "content": f"{brief}\n\n{CARRY_RULE}"}])
    return Usage.from_response(reply), reply.choices[0].message.content

Then every step gets the same short prefix and nothing else. No step can grow, because nothing is
appended between steps.

In [6]:
def ask_with_carry(carry, question):
    """One step against a fixed digest. Independent of the chain length."""
    reply = client.chat.completions.create(
        model=MODEL, max_tokens=200,
        messages=[{"role": "system", "content": SYSTEM},
                  {"role": "user", "content": f"Run digest:\n{carry}\n\n{question}"}])
    return Usage.from_response(reply), reply.choices[0].message.content

The digest costs one extra call, so this has to be measured, not assumed. Seven calls against six.

In [7]:
carry_usage, carry = make_carry(BRIEF)
pruned, answers = [carry_usage], []

for question in STEPS:
    usage, text = ask_with_carry(carry, question)
    pruned.append(usage)
    answers.append(text)

before, after = summarise(naive), summarise(pruned)
print(f"digest: {carry}\n")
print(f"before: {before['calls']} calls, {before['prompt_tokens']:5} prompt tokens, "
      f"${before['usd']:.6f}")
print(f"after : {after['calls']} calls, {after['prompt_tokens']:5} prompt tokens, "
      f"${after['usd']:.6f}")
print(f"saved : {1 - after['usd'] / before['usd']:.0%} of the bill")

digest: CMIP-X run 2041-B (24 members, 0.25 deg) shows 4 stalled members: 0, 7, 14, and 21. Worst drift is 0.21K. Most members are running well with increasing wall times and check-pointing intervals.

before: 6 calls,  6313 prompt tokens, $0.000770
after : 7 calls,  1418 prompt tokens, $0.000277
saved : 64% of the bill


A cheaper wrong answer is not a saving. Pruning is only safe when what you dropped was not needed,
and that is a property your code can check rather than hope for.

In [8]:
import re


def carry_is_safe(carry, must_keep):
    """Refuse a digest that lost something a later step depends on."""
    kept = set(re.findall(r"\d+", carry))
    missing = sorted(set(must_keep) - kept)
    if missing:
        raise ValueError(f"digest dropped members {missing}, do not prune this chain")
    return True


STALLED = [str(i) for i in range(24) if i % 7 == 0]
print(f"stalled members : {STALLED}")
print(f"digest keeps all: {carry_is_safe(carry, STALLED)}")
print(f"step 2 answered : {answers[1].strip()[:90]}")

stalled members : ['0', '7', '14', '21']
digest keeps all: True
step 2 answered : Based on your digest, the following member IDs need re-running:

*   **0**
*   **7**
*   *


## The gate

The regression is a digest prompt edited for length that quietly stops carrying the ids. This test
proves the guard refuses one, and it calls nothing.

In [9]:
def test_a_digest_that_drops_a_member_is_refused():
    try:
        carry_is_safe("members 0 and 7 stalled", ["0", "7", "14"])
    except ValueError:
        return
    raise AssertionError("a digest missing member 14 was accepted")


test_a_digest_that_drops_a_member_is_refused()
print("gate holds: a prune that loses a member id is refused")

gate holds: a prune that loses a member id is refused


Delete the raise from `carry_is_safe` and this test fails.

### Enterprise exploration

- The digest is one more call and one more thing that can be wrong. At what chain length does it
  start paying for itself, and how would you measure that per run?
- A digest is a lossy copy of evidence. If an operator disputes a handover note, what do you keep
  and for how long?
- Six steps here. What is the failure at sixty, and does it show up as a bill or as a refused
  request?
- The guard checks member ids because this chain needs ids. Who owns that list, and what happens the
  day a question needs a field the digest never kept?

### Key takeaways

- A chat call carries no memory. Every step pays for the whole list again.
- The prefix is charged once per call, not once per chain.
- Pass forward the smallest thing the next step needs, not everything you have.
- Prove the digest kept what later steps depend on, or the saving buys a wrong answer.